In [2]:
import pandas as pd

In [3]:
from pathlib import Path

ARQUIVO_HISTORICO = Path("brasileirao.csv")
df_historico = pd.read_csv(
    ARQUIVO_HISTORICO,
    encoding="utf-8-sig",
)

# Etapa 1 — Compreensão da base histórica

In [ ]:
df_historico = pd.read_csv(
    "brasileirao.csv",
    encoding="utf-8-sig",
)

print("Dimensão:", df_historico.shape)

print("\nTipos de dados:")
display(df_historico.dtypes.to_frame("tipo"))

print("\nPrimeiras linhas:")
display(df_historico.head())

print("\nEstatísticas:")
display(df_historico.describe(include="all"))

print("\nTemporadas:")
print(
    sorted(df_historico["season"].dropna().unique())
)

print("\nQuantidade de clubes:")
print(df_historico["team"].nunique())

print("\nClubes:")
display(
    pd.DataFrame(
        sorted(df_historico["team"].dropna().unique()),
        columns=["clubes"],
    )
)

## Etapa 1.1 — Dicionário de dados

In [4]:
dicionario_dados = pd.DataFrame(
    {
        "variavel": [
            "season",
            "place",
            "team",
            "points",
            "played",
            "won",
            "draw",
            "loss",
            "goals",
            "goals_taken",
            "goals_diff",
        ],
        "descricao": [
            "Temporada do Campeonato Brasileiro.",
            "Posição final do clube na classificação.",
            "Nome do clube.",
            "Pontuação final do clube.",
            "Quantidade de partidas disputadas.",
            "Quantidade de vitórias.",
            "Quantidade de empates.",
            "Quantidade de derrotas.",
            "Gols marcados.",
            "Gols sofridos.",
            "Saldo de gols.",
        ],
        "tipo_esperado": [
            "inteiro",
            "inteiro",
            "texto",
            "inteiro",
            "inteiro",
            "inteiro",
            "inteiro",
            "inteiro",
            "inteiro",
            "inteiro",
            "inteiro",
        ],
    }
)

display(dicionario_dados)

dicionario_dados.to_csv(
    PASTA_SAIDA / "dicionario_dados.csv",
    index=False,
    encoding="utf-8-sig",
)

,variavel,descricao,tipo_esperado
0,season,Temporada do Campeonato Brasileiro.,inteiro
1,place,Posição final do clube na classificação.,inteiro
2,team,Nome do clube.,texto
3,points,Pontuação final do clube.,inteiro
4,played,Quantidade de partidas disputadas.,inteiro
5,won,Quantidade de vitórias.,inteiro
6,draw,Quantidade de empates.,inteiro
7,loss,Quantidade de derrotas.,inteiro
8,goals,Gols marcados.,inteiro
9,goals_taken,Gols sofridos.,inteiro


# Etapa 2 — Qualidade e preparação da base histórica

In [5]:
COLUNAS_CLASSIFICACAO = [
    "season",
    "place",
    "team",
    "points",
    "played",
    "won",
    "draw",
    "loss",
    "goals",
    "goals_taken",
    "goals_diff",
]

df_historico = df_historico.copy()

for coluna in [
    "season",
    "place",
    "points",
    "played",
    "won",
    "draw",
    "loss",
    "goals",
    "goals_taken",
    "goals_diff",
]:
    df_historico[coluna] = pd.to_numeric(
        df_historico[coluna],
        errors="coerce",
    )

df_historico["team"] = (
    df_historico["team"]
    .astype("string")
    .str.strip()
)

df_historico["source"] = "kaggle_historico"

df_qualidade_historico = pd.DataFrame(
    {
        "verificacao": [
            "Registros totais",
            "Registros duplicados",
            "Valores ausentes",
            "Clubes únicos",
            "Temporadas únicas",
        ],
        "valor": [
            len(df_historico),
            df_historico.duplicated(
                subset=["season", "team"]
            ).sum(),
            int(df_historico.isna().sum().sum()),
            df_historico["team"].nunique(),
            df_historico["season"].nunique(),
        ],
    }
)

display(df_qualidade_historico)

,verificacao,valor
0,Registros totais,450
1,Registros duplicados,0
2,Valores ausentes,0
3,Clubes únicos,44
4,Temporadas únicas,22


#  Etapa 3 — Atualização dos dados por API

In [15]:
MAPA_CLUBES_API = {
    "Botafogo FR": "Botafogo",
    "CA Mineiro": "Atlético Mineiro",
    "CA Paranaense": "Athletico Paranaense",
    "Ceará SC": "Ceará",
    "Chapecoense AF": "Chapecoense",
    "Clube do Remo": "Remo",
    "Coritiba FBC": "Coritiba",
    "CR Flamengo": "Flamengo",
    "CR Vasco da Gama": "Vasco",
    "Cruzeiro EC": "Cruzeiro",
    "EC Bahia": "Bahia",
    "EC Juventude": "Juventude",
    "EC Vitória": "Vitória",
    "Fluminense FC": "Fluminense",
    "Fortaleza EC": "Fortaleza",
    "Grêmio FBPA": "Grêmio",
    "Mirassol FC": "Mirassol",
    "RB Bragantino": "Red Bull Bragantino",
    "Santos FC": "Santos",
    "SC Corinthians Paulista": "Corinthians",
    "SC Internacional": "Internacional",
    "SC Recife": "Sport",
    "SE Palmeiras": "Palmeiras",
    "São Paulo FC": "São Paulo",
}

arquivos_api = sorted(
    PASTA_API.glob("brasileirao_serie_a_20*.csv")
)

if not arquivos_api:
    raise FileNotFoundError(
        f"Não encontrei CSVs anuais em: {PASTA_API}"
    )

df_api = pd.concat(
    [
        pd.read_csv(arquivo, encoding="utf-8-sig")
        for arquivo in arquivos_api
    ],
    ignore_index=True,
)

df_api["data_partida"] = pd.to_datetime(
    df_api["data_partida"],
    errors="coerce",
    utc=True,
)

for coluna in [
    "match_id",
    "rodada",
    "temporada",
    "placar_mandante",
    "placar_visitante",
]:
    df_api[coluna] = pd.to_numeric(
        df_api[coluna],
        errors="coerce",
    )

df_api["time_mandante_canonico"] = (
    df_api["time_mandante"].replace(MAPA_CLUBES_API)
)

df_api["time_visitante_canonico"] = (
    df_api["time_visitante"].replace(MAPA_CLUBES_API)
)

df_api["source"] = "football_data_org"

display(df_api.tail(100))

,match_id,data_partida,status,rodada,temporada,time_mandante,time_visitante,placar_mandante,placar_visitante,vencedor,time_mandante_canonico,time_visitante_canonico,source
660,555022,2026-10-07 22:30:00+00:00,TIMED,29,2026,RB Bragantino,Mirassol FC,NaN,NaN,NaN,Red Bull Bragantino,Mirassol,football_data_org
661,555025,2026-10-07 22:30:00+00:00,TIMED,29,2026,SC Internacional,SC Corinthians Paulista,NaN,NaN,NaN,Internacional,Corinthians,football_data_org
662,555027,2026-10-07 22:30:00+00:00,TIMED,29,2026,Clube do Remo,Grêmio FBPA,NaN,NaN,NaN,Remo,Grêmio,football_data_org
663,555029,2026-10-07 23:00:00+00:00,TIMED,29,2026,EC Vitória,Chapecoense AF,NaN,NaN,NaN,Vitória,Chapecoense,football_data_org
664,555021,2026-10-07 23:30:00+00:00,TIMED,29,2026,Botafogo FR,CR Vasco da Gama,NaN,NaN,NaN,Botafogo,Vasco,football_data_org
...,...,...,...,...,...,...,...,...,...,...,...,...,...
755,555115,2026-12-02 00:00:00+00:00,SCHEDULED,38,2026,Grêmio FBPA,Mirassol FC,NaN,NaN,NaN,Grêmio,Mirassol,football_data_org
756,555116,2026-12-02 00:00:00+00:00,SCHEDULED,38,2026,SE Palmeiras,Coritiba FBC,NaN,NaN,NaN,Palmeiras,Coritiba,football_data_org
757,555117,2026-12-02 00:00:00+00:00,SCHEDULED,38,2026,Clube do Remo,SC Corinthians Paulista,NaN,NaN,NaN,Remo,Corinthians,football_data_org
758,555118,2026-12-02 00:00:00+00:00,SCHEDULED,38,2026,Santos FC,Botafogo FR,NaN,NaN,NaN,Santos,Botafogo,football_data_org


## Etapa 3.1 — Qualidade dos dados da API

In [7]:
df_qualidade_api = pd.DataFrame(
    {
        "verificacao": [
            "Registros totais",
            "Partidas finalizadas",
            "IDs duplicados",
            "Partidas sem placar final",
            "Datas inválidas",
            "Temporadas disponíveis",
        ],
        "valor": [
            len(df_api),
            int(df_api["status"].eq("FINISHED").sum()),
            int(df_api["match_id"].duplicated().sum()),
            int(
                (
                    df_api["status"].eq("FINISHED")
                    & (
                        df_api["placar_mandante"].isna()
                        | df_api["placar_visitante"].isna()
                    )
                ).sum()
            ),
            int(df_api["data_partida"].isna().sum()),
            ", ".join(
                map(
                    str,
                    sorted(
                        df_api["temporada"]
                        .dropna()
                        .astype(int)
                        .unique()
                    ),
                )
            ),
        ],
    }
)

display(df_qualidade_api)

,verificacao,valor
0,Registros totais,760
1,Partidas finalizadas,657
2,IDs duplicados,0
3,Partidas sem placar final,0
4,Datas inválidas,0
5,Temporadas disponíveis,"2025, 2026"


# Etapa 4 — Cálculo da classificação da API

In [8]:
def criar_classificacao_api(df_partidas):
    jogos = df_partidas[
        (df_partidas["status"] == "FINISHED")
        & df_partidas["placar_mandante"].notna()
        & df_partidas["placar_visitante"].notna()
    ].copy()

    mandantes = pd.DataFrame(
        {
            "season": jogos["temporada"],
            "team": jogos["time_mandante_canonico"],
            "played": 1,
            "won": (
                jogos["placar_mandante"]
                > jogos["placar_visitante"]
            ).astype(int),
            "draw": (
                jogos["placar_mandante"]
                == jogos["placar_visitante"]
            ).astype(int),
            "loss": (
                jogos["placar_mandante"]
                < jogos["placar_visitante"]
            ).astype(int),
            "goals": jogos["placar_mandante"],
            "goals_taken": jogos["placar_visitante"],
        }
    )

    visitantes = pd.DataFrame(
        {
            "season": jogos["temporada"],
            "team": jogos["time_visitante_canonico"],
            "played": 1,
            "won": (
                jogos["placar_visitante"]
                > jogos["placar_mandante"]
            ).astype(int),
            "draw": (
                jogos["placar_visitante"]
                == jogos["placar_mandante"]
            ).astype(int),
            "loss": (
                jogos["placar_visitante"]
                < jogos["placar_mandante"]
            ).astype(int),
            "goals": jogos["placar_visitante"],
            "goals_taken": jogos["placar_mandante"],
        }
    )

    classificacao = pd.concat(
        [mandantes, visitantes],
        ignore_index=True,
    )

    classificacao = (
        classificacao
        .groupby(["season", "team"], as_index=False)
        .agg(
            played=("played", "sum"),
            won=("won", "sum"),
            draw=("draw", "sum"),
            loss=("loss", "sum"),
            goals=("goals", "sum"),
            goals_taken=("goals_taken", "sum"),
        )
    )

    classificacao["points"] = (
        classificacao["won"] * 3
        + classificacao["draw"]
    )

    classificacao["goals_diff"] = (
        classificacao["goals"]
        - classificacao["goals_taken"]
    )

    classificacao = classificacao.sort_values(
        ["season", "points", "won", "goals_diff", "goals"],
        ascending=[True, False, False, False, False],
    )

    classificacao["place"] = (
        classificacao
        .groupby("season")
        .cumcount()
        .add(1)
    )

    classificacao["source"] = "football_data_org"

    return classificacao


df_api_classificacao = criar_classificacao_api(df_api)

display(df_api_classificacao.head())

,season,team,played,won,draw,loss,goals,goals_taken,points,goals_diff,place,source
6,2025,Flamengo,38,23,10,5,78.0,27.0,79,51.0,1,football_data_org
13,2025,Palmeiras,38,23,7,8,66.0,33.0,76,33.0,2,football_data_org
5,2025,Cruzeiro,38,19,13,6,55.0,31.0,70,24.0,3,football_data_org
12,2025,Mirassol,38,18,13,7,63.0,39.0,67,24.0,4,football_data_org
7,2025,Fluminense,38,19,7,12,50.0,39.0,64,11.0,5,football_data_org


## Etapa 4.1 — Integração histórica e API

In [9]:
df_base_consolidada = pd.concat(
    [
        df_historico[
            COLUNAS_CLASSIFICACAO + ["source"]
        ],
        df_api_classificacao[
            COLUNAS_CLASSIFICACAO + ["source"]
        ],
    ],
    ignore_index=True,
)

df_base_consolidada = (
    df_base_consolidada
    .sort_values(["season", "place"])
    .reset_index(drop=True)
)

df_base_consolidada.to_csv(
    PASTA_SAIDA / "brasileirao_classificacao_2003_2026.csv",
    index=False,
    encoding="utf-8-sig",
)

print("Base consolidada:", df_base_consolidada.shape)

display(df_base_consolidada.tail(20))

Base consolidada: (490, 12)


,season,place,team,points,played,won,draw,loss,goals,goals_taken,goals_diff,source
470,2026,1,Flamengo,60,28,18,6,4,55.0,23.0,32.0,football_data_org
471,2026,2,Palmeiras,57,28,16,9,3,47.0,21.0,26.0,football_data_org
472,2026,3,Athletico Paranaense,49,28,14,7,7,43.0,32.0,11.0,football_data_org
473,2026,4,Fluminense,48,28,13,9,6,44.0,36.0,8.0,football_data_org
474,2026,5,Bahia,46,28,12,10,6,43.0,35.0,8.0,football_data_org
475,2026,6,Cruzeiro,45,28,13,6,9,42.0,40.0,2.0,football_data_org
476,2026,7,Atlético Mineiro,40,27,11,7,9,36.0,32.0,4.0,football_data_org
477,2026,8,Santos,38,27,10,8,9,41.0,40.0,1.0,football_data_org
478,2026,9,Coritiba,38,28,10,8,10,37.0,43.0,-6.0,football_data_org
479,2026,10,Red Bull Bragantino,36,27,10,6,11,33.0,31.0,2.0,football_data_org


# Etapa 8.1 — Regras históricas e correções

In [10]:
from IPython.display import HTML, display

TITULOS_POR_ANO = {
    2003: {"Copa do Brasil": "Cruzeiro", "Libertadores": None, "Sul-Americana": None},
    2004: {"Copa do Brasil": "Santo André", "Libertadores": None, "Sul-Americana": None},
    2005: {"Copa do Brasil": "Paulista", "Libertadores": "São Paulo", "Sul-Americana": None},
    2006: {"Copa do Brasil": "Flamengo", "Libertadores": "Internacional", "Sul-Americana": None},
    2007: {"Copa do Brasil": "Fluminense", "Libertadores": None, "Sul-Americana": None},
    2008: {"Copa do Brasil": "Sport", "Libertadores": None, "Sul-Americana": "Internacional"},
    2009: {"Copa do Brasil": "Corinthians", "Libertadores": None, "Sul-Americana": None},
    2010: {"Copa do Brasil": "Santos", "Libertadores": "Internacional", "Sul-Americana": None},
    2011: {"Copa do Brasil": "Vasco", "Libertadores": "Santos", "Sul-Americana": None},
    2012: {"Copa do Brasil": "Palmeiras", "Libertadores": "Corinthians", "Sul-Americana": "São Paulo"},
    2013: {"Copa do Brasil": "Flamengo", "Libertadores": "Atlético Mineiro", "Sul-Americana": None},
    2014: {"Copa do Brasil": "Atlético Mineiro", "Libertadores": None, "Sul-Americana": None},
    2015: {"Copa do Brasil": "Palmeiras", "Libertadores": None, "Sul-Americana": None},
    2016: {"Copa do Brasil": "Grêmio", "Libertadores": None, "Sul-Americana": "Chapecoense"},
    2017: {"Copa do Brasil": "Cruzeiro", "Libertadores": "Grêmio", "Sul-Americana": None},
    2018: {"Copa do Brasil": "Cruzeiro", "Libertadores": None, "Sul-Americana": "Athletico Paranaense"},
    2019: {"Copa do Brasil": "Athletico Paranaense", "Libertadores": "Flamengo", "Sul-Americana": None},
    2020: {"Copa do Brasil": "Palmeiras", "Libertadores": "Palmeiras", "Sul-Americana": None},
    2021: {"Copa do Brasil": "Atlético Mineiro", "Libertadores": "Palmeiras", "Sul-Americana": "Athletico Paranaense"},
    2022: {"Copa do Brasil": "Flamengo", "Libertadores": "Flamengo", "Sul-Americana": None},
    2023: {"Copa do Brasil": "São Paulo", "Libertadores": "Fluminense", "Sul-Americana": None},
    2024: {"Copa do Brasil": "Flamengo", "Libertadores": "Botafogo", "Sul-Americana": None},
    2025: {"Copa do Brasil": "Corinthians", "Libertadores": "Flamengo", "Sul-Americana": None},
    2026: {"Copa do Brasil": None, "Libertadores": None, "Sul-Americana": None},
}

REGRAS_BASE = {
    2003: {"direta": 4, "pre": 0, "sula": 6, "rebaixados": 2},
    2004: {"direta": 3, "pre": 1, "sula": 8, "rebaixados": 4},
    2005: {"direta": 2, "pre": 2, "sula": 8, "rebaixados": 4},
    2006: {"direta": 3, "pre": 2, "sula": 7, "rebaixados": 4},
    2007: {"direta": 3, "pre": 1, "sula": 8, "rebaixados": 4},
}

for ano in range(2008, 2017):
    REGRAS_BASE[ano] = {
        "direta": 3,
        "pre": 1,
        "sula": 8,
        "rebaixados": 4,
    }

for ano in range(2017, 2027):
    REGRAS_BASE[ano] = {
        "direta": 4,
        "pre": 2,
        "sula": 6,
        "rebaixados": 4,
    }

for ano in [2012, 2013, 2014, 2015]:
    REGRAS_BASE[ano]["sula"] = 0

CORRECOES_GE = {
    2003: {"Libertadores": ["Coritiba"]},
    2010: {"Pré-Libertadores": ["Corinthians"]},
    2011: {
        "Pré-Libertadores": ["Internacional"],
        "Sul-Americana": ["Bahia"],
    },
    2016: {"Libertadores": ["Atlético Mineiro"]},
    2017: {
        "Libertadores": ["Flamengo"],
        "Pré-Libertadores": ["Vasco", "Chapecoense"],
        "Sul-Americana": ["São Paulo", "Fluminense"],
    },
    2019: {
        "Libertadores": ["São Paulo"],
        "Pré-Libertadores": ["Internacional", "Corinthians"],
        "Sul-Americana": ["Atlético Mineiro", "Fluminense"],
    },
    2020: {
        "Libertadores": ["Fluminense"],
        "Pré-Libertadores": ["Santos"],
        "Sul-Americana": ["Bahia"],
    },
    2021: {
        "Libertadores": ["Corinthians", "Red Bull Bragantino"],
        "Sul-Americana": [
            "Fluminense",
            "Atlético Mineiro",
            "São Paulo",
            "Cuiabá",
        ],
    },
    2022: {
        "Libertadores": ["Athletico Paranaense"],
        "Sul-Americana": [
            "Atlético Mineiro",
            "Fortaleza",
            "Goiás",
            "Red Bull Bragantino",
        ],
    },
    2024: {
        "Libertadores": ["Internacional", "São Paulo"],
        "Pré-Libertadores": ["Corinthians", "Bahia"],
        "Sul-Americana": ["Fluminense", "Grêmio"],
    },
    2025: {
        "Libertadores": ["Fluminense"],
        "Pré-Libertadores": ["Botafogo", "Bahia"],
        "Sul-Americana": ["Vasco"],
    },
}

SEM_CLASSIFICACAO = {
    2005: ["Fortaleza"],
}

## Etapa 8.2 — Criação da tabela do campeonato

In [11]:
def criar_tabela_campeonato(df_classificacao, temporada):
    tabela = (
        df_classificacao
        .query("season == @temporada")
        .copy()
        .sort_values("place")
        .reset_index(drop=True)
    )

    if tabela.empty:
        raise ValueError(f"Não há dados para a temporada {temporada}.")

    regra = REGRAS_BASE[temporada]
    titulos = TITULOS_POR_ANO.get(temporada, {})

    tabela["Situação"] = "Meio de tabela"
    tabela["Origem da vaga"] = ""

    total_times = len(tabela)

    tabela.loc[
        tabela["place"] > total_times - regra["rebaixados"],
        "Situação",
    ] = "Rebaixamento"

    vagas_diretas = tabela[
        tabela["place"] <= regra["direta"]
    ].index

    tabela.loc[vagas_diretas, "Situação"] = "Libertadores"
    tabela.loc[vagas_diretas, "Origem da vaga"] = "Brasileirão"

    vagas_pre = tabela[
        (tabela["place"] > regra["direta"])
        & (
            tabela["place"]
            <= regra["direta"] + regra["pre"]
        )
    ].index

    tabela.loc[vagas_pre, "Situação"] = "Pré-Libertadores"
    tabela.loc[vagas_pre, "Origem da vaga"] = "Brasileirão"

    for origem, clube in titulos.items():
        if clube is not None:
            mascara = tabela["team"].eq(clube)

            tabela.loc[mascara, "Situação"] = "Libertadores"
            tabela.loc[mascara, "Origem da vaga"] = origem

    ja_classificados = tabela["Situação"].isin(
        ["Libertadores", "Pré-Libertadores"]
    )

    vagas_sula = tabela[
        ~ja_classificados
        & ~tabela["Situação"].eq("Rebaixamento")
    ].head(regra["sula"]).index

    tabela.loc[vagas_sula, "Situação"] = "Sul-Americana"
    tabela.loc[vagas_sula, "Origem da vaga"] = "Brasileirão"

    for clube in SEM_CLASSIFICACAO.get(temporada, []):
        mascara = tabela["team"].eq(clube)

        tabela.loc[mascara, "Situação"] = "Meio de tabela"
        tabela.loc[mascara, "Origem da vaga"] = ""

    for situacao, clubes in CORRECOES_GE.get(
        temporada,
        {},
    ).items():
        for clube in clubes:
            mascara = tabela["team"].eq(clube)

            tabela.loc[mascara, "Situação"] = situacao
            tabela.loc[
                mascara,
                "Origem da vaga",
            ] = "Regra histórica confirmada"

    tabela["aproveitamento"] = (
        tabela["points"]
        .div(tabela["played"] * 3)
        .mul(100)
        .fillna(0)
        .round(1)
    )

    tabela = tabela.rename(
        columns={
            "place": "Pos.",
            "team": "Time",
            "points": "Pts",
            "played": "J",
            "won": "V",
            "draw": "E",
            "loss": "D",
            "goals": "GP",
            "goals_taken": "GC",
            "goals_diff": "SG",
            "aproveitamento": "Aproveitamento (%)",
        }
    )

    return tabela[
        [
            "Pos.",
            "Time",
            "Pts",
            "J",
            "V",
            "E",
            "D",
            "GP",
            "GC",
            "SG",
            "Aproveitamento (%)",
            "Situação",
            "Origem da vaga",
        ]
    ]

In [12]:
def exibir_tabela_ge(tabela):
    tabela_html = tabela.copy()

    cores = {
        "Libertadores": "#198754",
        "Pré-Libertadores": "#0d6efd",
        "Sul-Americana": "#d39e00",
        "Rebaixamento": "#dc3545",
        "Meio de tabela": "#6c757d",
    }

    tabela_html["Pos."] = tabela_html.apply(
        lambda linha: (
            f'<span style="display:inline-block;width:28px;'
            f'padding:4px;text-align:center;color:white;'
            f'font-weight:bold;border-radius:4px;'
            f'background:{cores[linha["Situação"]]}">'
            f'{linha["Pos."]}</span>'
        ),
        axis=1,
    )

    tabela_html["Situação"] = tabela_html["Situação"].apply(
        lambda situacao: (
            f'<strong style="color:{cores[situacao]}">'
            f'{situacao}</strong>'
        )
    )

    tabela_html["Aproveitamento (%)"] = (
        tabela_html["Aproveitamento (%)"]
        .map(lambda valor: f"{valor:.1f}%")
    )

    html = tabela_html.to_html(
        index=False,
        escape=False,
        classes="tabela-ge",
    )

    estilo = """
    <style>
        .tabela-ge {
            border-collapse: collapse;
            font-family: Arial, sans-serif;
            font-size: 14px;
            width: 100%;
        }

        .tabela-ge th {
            background: #222;
            color: white;
            padding: 10px;
            text-align: center;
        }

        .tabela-ge td {
            padding: 8px;
            border-bottom: 1px solid #ddd;
            text-align: center;
        }

        .tabela-ge td:nth-child(2) {
            text-align: left;
            font-weight: bold;
        }

        .tabela-ge tr:hover {
            background-color: #f5f5f5;
        }
    </style>
    """

    display(HTML(estilo + html))

In [13]:
TEMPORADA_EXIBIDA = 2026

tabela_ge = criar_tabela_campeonato(
    df_base_consolidada,
    TEMPORADA_EXIBIDA,
)

tabela_ge.to_csv(
    PASTA_SAIDA / f"tabela_ge_{TEMPORADA_EXIBIDA}.csv",
    index=False,
    encoding="utf-8-sig",
)

print(f"CAMPEONATO BRASILEIRO SÉRIE A — {TEMPORADA_EXIBIDA}")
exibir_tabela_ge(tabela_ge)

CAMPEONATO BRASILEIRO SÉRIE A — 2026


Pos.,Time,Pts,J,V,E,D,GP,GC,SG,Aproveitamento (%),Situação,Origem da vaga
1,Flamengo,60,28,18,6,4,55.0,23.0,32.0,71.4%,Libertadores,Brasileirão
2,Palmeiras,57,28,16,9,3,47.0,21.0,26.0,67.9%,Libertadores,Brasileirão
3,Athletico Paranaense,49,28,14,7,7,43.0,32.0,11.0,58.3%,Libertadores,Brasileirão
4,Fluminense,48,28,13,9,6,44.0,36.0,8.0,57.1%,Libertadores,Brasileirão
5,Bahia,46,28,12,10,6,43.0,35.0,8.0,54.8%,Pré-Libertadores,Brasileirão
6,Cruzeiro,45,28,13,6,9,42.0,40.0,2.0,53.6%,Pré-Libertadores,Brasileirão
7,Atlético Mineiro,40,27,11,7,9,36.0,32.0,4.0,49.4%,Sul-Americana,Brasileirão
8,Santos,38,27,10,8,9,41.0,40.0,1.0,46.9%,Sul-Americana,Brasileirão
9,Coritiba,38,28,10,8,10,37.0,43.0,-6.0,45.2%,Sul-Americana,Brasileirão
10,Red Bull Bragantino,36,27,10,6,11,33.0,31.0,2.0,44.4%,Sul-Americana,Brasileirão
